# Experiments and export
Evaluation protocols (RQ1), calibration (RQ3), INT8 (RQ2) and the header file for the ESP32.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/mshahnaza/emg-cross-session-mcu.git /content/emg-thesis 2>/dev/null || git -C /content/emg-thesis pull
!pip -q install -r /content/emg-thesis/requirements.txt
import sys
sys.path.append('/content/emg-thesis/src')

In [ ]:
from emg.config import test_sessions
from emg.data import load_features
from emg.experiments import rq1, train_calibration_models, renorm_table, board_methods_table
from emg.quantization import quantize_mlp, model_sizes
from emg.export import write_header

SUBJECT = 1
FEATURES_PATH = f'/content/drive/MyDrive/ninapro/features_S{SUBJECT}.npz'
RESULTS_DIR = '/content/drive/MyDrive/ninapro/results/'
!mkdir -p {RESULTS_DIR}

data, extra = load_features(FEATURES_PATH, SUBJECT)
print('Channels:', len(extra['channels']))

In [ ]:
rq1_table, standard, pool = rq1(data, SUBJECT)
rq1_table

In [ ]:
models, X_sess = train_calibration_models(data, SUBJECT)
renorm_df, renorm_summary = renorm_table(data, SUBJECT, standard, models, pool)
renorm_summary

In [ ]:
q = quantize_mlp(models['MLP'], X_sess)
size_float, size_int8 = model_sizes(q)
print('Model size float32:', size_float, 'bytes, INT8:', size_int8, 'bytes')

In [ ]:
board_df, board_summary = board_methods_table(data, SUBJECT, q, pool)
board_summary

In [ ]:
exp = write_header('model_data.h', q, pool, data[test_sessions(SUBJECT)[-1]], extra)
for name, (af, ai) in exp.items():
    print(f'{name}: float32 {af:.2f} / INT8 {ai:.2f}')

In [ ]:
rq1_table.to_csv(RESULTS_DIR + f'rq1_S{SUBJECT}.csv', index=False)
renorm_df.to_csv(RESULTS_DIR + f'renorm_S{SUBJECT}.csv', index=False)
board_df.to_csv(RESULTS_DIR + f'board_methods_S{SUBJECT}.csv', index=False)
from google.colab import files
files.download('model_data.h')